# The reference $\lambda^2_0$

RM-synthesis derotates every channel to a common reference before transforming
(Brentjens & de Bruyn 2005, eq. 25):

$$\tilde{F}(\phi) = K \int \tilde{P}(\lambda^2)\, e^{-2i\phi(\lambda^2 - \lambda^2_0)} \, \mathrm{d}\lambda^2$$

That reference is a *choice*, not a fitted quantity: the shift theorem means it
moves phase only, never amplitude. B&dB pick it by nulling the derivative of the
RMSF's orthogonal (imaginary) response at $\phi = 0$, which gives eq. 32:

$$\lambda^2_0 = \frac{\int W(\lambda^2)\lambda^2 \mathrm{d}\lambda^2}{\int W(\lambda^2)\mathrm{d}\lambda^2}$$

the weighted mean of the observed $\lambda^2$. The point of doing so is that
*"the response in the entire main peak of the RMTF should be parallel to the
actual polarization vector at $\lambda_0$"*.

$W(\lambda^2)$ is the observation's own weight function, so when weighting or
flagging varies across an image, so does the ideal reference. `rm-lite` lets you
choose with `lam_sq_0_m2`:

| value | meaning |
| --- | --- |
| `"auto"` (default) | one weighted mean for the whole dataset |
| a number in m$^2$ | a reference you pin, e.g. to match another cube |
| `"per_pixel"` | each pixel's own weighted mean |

In every case a `lam_sq_0_map` comes back, so you can always move between
references afterwards.

In [ ]:
from __future__ import annotations

import dask.array as da
import numpy as np

from rm_lite.tools_3d.rmsynth import rmsynth_3d
from rm_lite.utils.logging import quiet_logs
from rm_lite.utils.synthesis import derotate_to, freq_to_lambda2, lambda2_to_freq

rng = np.random.default_rng(2026)
n_freq, ny, nx = 96, 6, 6
freq_arr_hz = np.linspace(744e6, 1032e6, n_freq)
lambda_sq = freq_to_lambda2(freq_arr_hz)

A mosaicked field makes the reference genuinely position dependent. The primary
beam shrinks as $1/\nu$, so a pixel near the edge is inside the beam at the
bottom of the band and outside it at the top: the channels contributing vary
across the image.

In [ ]:
yy, xx = np.mgrid[0:ny, 0:nx]
radius = np.sqrt((yy - (ny - 1) / 2) ** 2 + (xx - (nx - 1) / 2) ** 2)
radius = 0.78 * radius / radius.max()
inside = radius[None] <= (freq_arr_hz[0] / freq_arr_hz)[:, None, None]

angle = 2 * (
    rng.uniform(-60, 60, (ny, nx))[None] * lambda_sq[:, None, None]
    + rng.uniform(0, np.pi, (ny, nx))[None]
)
stokes_q = np.where(inside, 0.6 * np.cos(angle), np.nan)
stokes_u = np.where(inside, 0.6 * np.sin(angle), np.nan)
# linmos blanks the noise cube the same way it blanks the data.
weight_arr = np.where(inside, 1.0 / 1e-3**2, np.nan)

n_chan = inside.sum(axis=0)
print(f"channels kept: centre {n_chan[ny // 2, nx // 2]}, corner {n_chan[0, 0]}, of {n_freq}")
assert n_chan.min() < n_chan.max()


def synth(lam_sq_0_m2, **kwargs):
    with quiet_logs():
        return rmsynth_3d(
            da.from_array(stokes_q, chunks=(-1, 3, 3)),
            da.from_array(stokes_u, chunks=(-1, 3, 3)),
            freq_arr_hz,
            weight_arr=weight_arr,
            lam_sq_0_m2=lam_sq_0_m2,
            phi_max_radm2=200.0,
            d_phi_radm2=2.0,
            **kwargs,
        )

## One reference for the cube

`"auto"` is B&dB eq. 32 over the whole dataset. The map comes back constant.

In [ ]:
auto = synth("auto")
auto_map = auto.lam_sq_0_map.compute()
print(f"lam_sq_0 = {auto.lam_sq_0_m2:.6f} m^2, map spans "
      f"{auto_map.min():.6f} to {auto_map.max():.6f}")
assert np.allclose(auto_map, auto.lam_sq_0_m2)

## A reference you pin

Give a value in m$^2$ to share a reference with another cube, so the two FDFs
can be compared without derotating either.

In [ ]:
pinned = synth(0.1)
assert pinned.lam_sq_0_m2 == 0.1
assert np.allclose(pinned.lam_sq_0_map.compute(), 0.1)
print(f"pinned to {pinned.lam_sq_0_m2} m^2")

## A reference per pixel

`"per_pixel"` applies eq. 32 to each pixel's own weights and flagging. Edge
pixels lost the top of the band, so their weighted mean $\lambda^2$ is larger.

This also switches on the per-pixel RMSF cube: a pixel's RMSF has to sit at the
same reference as its FDF, or RM-CLEAN subtracts a response rotated away from
the components it is fitting.

In [ ]:
per_pixel = synth("per_pixel")
pixel_map = per_pixel.lam_sq_0_map.compute()
print(f"map spans {pixel_map.min():.6f} to {pixel_map.max():.6f} m^2")
assert pixel_map.max() > pixel_map.min()
assert per_pixel.rmsf_cube is not None

## The flux reference follows the phase reference

The Stokes I model's reference frequency is derived from $\lambda^2_0$ in one
place, so the frequency the FDF is derotated to and the frequency the Stokes I
terms are defined at are the same quantity, in every mode.

In [ ]:
for label, result in (("auto", auto), ("pinned", pinned), ("per_pixel", per_pixel)):
    ref_freq = result.stokes_i_ref_freq_hz
    if ref_freq is None:  # no Stokes I model was supplied here
        ref_freq = lambda2_to_freq(result.lam_sq_0_map)
    ref_freq_arr = np.asarray(
        ref_freq.compute() if isinstance(ref_freq, da.Array) else ref_freq
    )
    lam_map = result.lam_sq_0_map.compute()
    np.testing.assert_allclose(
        np.broadcast_to(ref_freq_arr, lam_map.shape), lambda2_to_freq(lam_map),
        rtol=1e-12,
    )
    print(f"{label:10s} reference frequency matches lambda^2_0 everywhere")

## Moving between references

Because eq. 25 is a shift theorem, changing the reference is an exact phase
ramp: `derotate_to` moves an FDF (or an RMSF) from one reference to another
without touching amplitudes, and inverts itself. That is what the map is for.

In [ ]:
auto_fdf, pixel_fdf = da.compute(auto.fdf_dirty_cube, per_pixel.fdf_dirty_cube)

# Amplitudes do not depend on the reference at all.
np.testing.assert_allclose(np.abs(auto_fdf), np.abs(pixel_fdf), rtol=1e-14)

# And the per-pixel FDF is exactly the shared one, moved.
moved = derotate_to(auto_fdf, auto.phi_arr_radm2, auto.lam_sq_0_m2, pixel_map)
np.testing.assert_allclose(moved, pixel_fdf, rtol=1e-12, atol=1e-14)

back = derotate_to(pixel_fdf, auto.phi_arr_radm2, pixel_map, auto.lam_sq_0_m2)
np.testing.assert_allclose(back, auto_fdf, rtol=1e-12, atol=1e-14)
print("derotation is exact and reversible")

## Why it matters

B&dB choose the weighted mean to keep the RMSF's orthogonal response near zero
across the main lobe, warning that otherwise *"if the Faraday depth of a frame
is only a tenth of the width of the RMTF away from the actual Faraday depth of
the source, the (real, imaginary) vector may already be rotated by several tens
of degrees"*.

For an edge pixel that lost part of the band, the cube-wide reference is no
longer its weighted mean, and its orthogonal response grows.

In [ ]:
from rm_lite.utils.synthesis import get_fwhm_rmsf, get_rmsf_nufft

corner = inside[:, 0, 0]
weights = np.where(corner, 1.0, 0.0)
own_lam_sq_0 = float(np.sum(weights * lambda_sq) / np.sum(weights))
fwhm = get_fwhm_rmsf(lambda_sq[corner]).fwhm_rmsf_radm2

for label, lam_sq_0 in (("cube-wide", auto.lam_sq_0_m2), ("its own", own_lam_sq_0)):
    with quiet_logs():
        rmsf = np.asarray(
            get_rmsf_nufft(
                lambda_sq, auto.phi_arr_radm2, weights, lam_sq_0, do_fit_rmsf=False
            ).rmsf_cube
        ).ravel()
    rmsf = rmsf / np.abs(rmsf).max()
    lobe = np.abs(auto.phi_double_arr_radm2) <= fwhm / 2
    print(f"{label:10s} reference: peak |Im RMSF| in the main lobe "
          f"{np.abs(rmsf.imag[lobe]).max():.3f}")
    if label == "its own":
        own_response = np.abs(rmsf.imag[lobe]).max()
    else:
        cube_response = np.abs(rmsf.imag[lobe]).max()

assert own_response < cube_response
print("\nthe pixel's own reference keeps the response parallel, as eq. 32 intends")

## Choosing

- `"auto"` is the right default. Angle maps are directly comparable across the
  image, which is what you want when the band is the same everywhere.
- `"per_pixel"` is what B&dB's criterion asks for when weighting or flagging
  varies spatially, as in a mosaic. It costs the per-pixel RMSF cube, and the
  Stokes I terms become each pixel's own, so compare `alpha` across the image by
  re-evaluating to a common frequency first.
- A pinned value shares a reference with another dataset.

Whichever you pick, `lam_sq_0_map` and `derotate_to` let you move afterwards. To
recover the polarisation angle at $\lambda^2 = 0$, B&dB eq. 33 is
$\chi_0 = \chi(\lambda^2_0) - \phi\lambda^2_0$.